In [1]:
from utils import load_splits, tokenize_dataset, compute_metrics, classification_metrics
import torch
import pandas as pd

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [3]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

In [4]:
df_test = pd.merge(X_test, y_test, left_index=True, right_index=True)
df_test

,title,labels
0,Nigeria’s Battles With Boko Haram Scar the Lan...,0
1,REMEMBER DEATH PANELS IN OBAMACARE? THEY’RE BA...,1
2,Missouri Republican Calls Getting Pregnant The...,1
3,"Families To Roof: 'I Forgive You, And Have Mer...",0
4,Chicago Paper: Make All Universities ’Sanctuar...,0
...,...,...
9192,Ex-FBI chief Mueller named special prosecutor ...,0
9193,BUSTED! HERE’S WHY HILLARY CLINTON’S Brother-I...,1
9194,“BIGGER THAN WATERGATE”: Sara Carter’s Take On...,1
9195,Wal-Mart reopens most Gulf region stores after...,0


In [5]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [2]:
import json
from pathlib import Path
logs = {}
for file in Path("logs").glob("*.json"):
    run = json.loads(file.read_text())
    # Accept the per-epoch list format produced by the training notebook.
    if isinstance(run, list) and run:
        model_name, learning_rate = file.stem.rsplit("_", 1)
        run_dir = Path("models") / file.stem
        state_path = run_dir / "trainer_state.json"
        best_epoch = max(run, key=lambda entry: entry["eval_f1_macro"])
        checkpoint = run_dir / f"checkpoint-{best_epoch['step']}"
        if state_path.is_file():
            state = json.loads(state_path.read_text())
            if state.get("best_model_checkpoint"):
                checkpoint = run_dir / Path(state["best_model_checkpoint"]).name
        run = {
            "model_name": model_name,
            "learning_rate": float(learning_rate),
            "best_checkpoint": str(checkpoint),
            "metrics": run,
            "log_history": run,
        }
    if (isinstance(run, dict) and run.get("best_checkpoint")
            and run.get("status", "completed") == "completed" and run.get("metrics")):
        logs[file.stem] = run
if not logs:
    raise ValueError("Retrain models on the corrected splits to generate run metadata before evaluation.")


In [3]:
logs

{'bert-base-uncased_1e-05': {'model_name': 'bert-base-uncased',
  'learning_rate': 1e-05,
  'best_checkpoint': 'models/bert-base-uncased_1e-05/checkpoint-6710',
  'metrics': [{'epoch': 1.0,
    'grad_norm': 8.366775512695312,
    'learning_rate': 1e-05,
    'loss': 0.32098083950901174,
    'step': 1342,
    'eval_accuracy': 0.9008372295313689,
    'eval_f1_macro': 0.8989982362275906,
    'eval_log_loss': 0.2413845956325531,
    'eval_loss': 0.24138455092906952,
    'eval_precision_macro': 0.9014461667981803,
    'eval_recall_macro': 0.8972433314546966,
    'eval_runtime': 9.4647,
    'eval_samples_per_second': 971.716,
    'eval_steps_per_second': 30.429},
   {'epoch': 2.0,
    'grad_norm': 53.60539627075195,
    'learning_rate': 1e-05,
    'loss': 0.19746116945178604,
    'step': 2684,
    'eval_accuracy': 0.9040991627704686,
    'eval_f1_macro': 0.9019065959947075,
    'eval_log_loss': 0.25243812799453735,
    'eval_loss': 0.2524404525756836,
    'eval_precision_macro': 0.90715823866

In [8]:
val_metrics = []
for run in logs.values():
    history = run.get("log_history", run["metrics"])
    training_by_epoch = {
        entry["epoch"]: entry for entry in history
        if "epoch" in entry and "loss" in entry
    }
    for epoch in run["metrics"]:
        training = training_by_epoch.get(epoch["epoch"], {})
        val_metrics.append({**training, **epoch, "model": run["model_name"],
                            "learning_rate": run["learning_rate"]})


In [ ]:
val = pd.DataFrame(val_metrics)

if "loss" not in val:
    val["loss"] = float("nan")

val = val.sort_values("model").reset_index(drop=True)

val = val.drop(columns=["grad_norm", "step", "eval_runtime", "eval_samples_per_second", "eval_steps_per_second"], errors="ignore")

val

In [ ]:
val["display_name"] = (
    val["model"].map(lambda x: x[0]) +"_lr_" + val["learning_rate"].map(lambda x: format(x, ".0e"))
    + "_e_" + val["epoch"].astype("Int64").astype(str)
).fillna("baseline")

In [ ]:
import matplotlib.pyplot as plt

metrics = ['eval_log_loss', 'eval_accuracy', 'eval_precision_macro', 'eval_recall_macro', 'eval_f1_macro']



for index, metric in enumerate(metrics):
    fig, axes = plt.subplots(figsize = (10,5))

    axes.bar(val["display_name"], val[metric])
    axes.set_title(metric)
    axes.set_ylim(0,1)
    axes.set_ylabel('Scores')
    axes.tick_params(axis="x", labelrotation=90)
    plt.tight_layout()
    plt.show()



In [ ]:
loss_val = val.dropna(subset=["loss", "eval_loss"]).copy()
loss_val = loss_val.sort_values(["model", "epoch"])

groups = list(loss_val.groupby(["model", "learning_rate"], sort=True))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for ax, (model_name, group) in zip(axes, groups):
    ax.plot(group["epoch"], group["loss"], marker="o", label="Training loss")
    ax.plot(group["epoch"], group["eval_loss"], marker="o", label="Validation loss")
    ax.set_title(model_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)
    ax.legend()

for ax in axes[len(groups):]:
    ax.set_visible(False)

fig.suptitle("Training vs. validation loss")
plt.tight_layout()
plt.show()


In [ ]:
f1_macro_val = val[val["model"]=="bert-base-uncased"].dropna(subset=["loss", "eval_loss"]).copy()
f1_macro_val = f1_macro_val.sort_values(["model", "epoch"])

groups = list(f1_macro_val.groupby(["model", "learning_rate"], sort=True))
fig, ax = plt.subplots(figsize=(15, 8))


for (model_name, group) in groups:
    ax.plot(group["epoch"], group["eval_f1_macro"], marker="o", label=f"{model_name}")
ax.set_title('BERT F1-macro curve over training epochs')
ax.set_xlabel("Epoch")
ax.set_ylabel("F1-macro")
ax.grid(alpha=0.3)
ax.legend()

for ax in axes[len(groups):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
f1_macro_val = val[val["model"]=="distilbert-base-uncased"].dropna(subset=["loss", "eval_loss"]).copy()
f1_macro_val = f1_macro_val.sort_values(["model", "epoch"])

groups = list(f1_macro_val.groupby(["model", "learning_rate"], sort=True))
fig, ax = plt.subplots(figsize=(15, 8))


for (model_name, group) in groups:
    ax.plot(group["epoch"], group["eval_f1_macro"], marker="o", label=f"{model_name}")
ax.set_title('DistilBERT F1-macro curve over training epochs')
ax.set_xlabel("Epoch")
ax.set_ylabel("F1-macro")
ax.grid(alpha=0.3)
ax.legend()

for ax in axes[len(groups):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
distilbert = val[val["model"] == "distilbert-base-uncased"]
best_distil =distilbert[(distilbert["eval_f1_macro"]==distilbert["eval_f1_macro"].max())]
bert = val[val["model"] == "bert-base-uncased"]
best_bert = bert[(bert["eval_f1_macro"]==bert["eval_f1_macro"].max())]

selected = pd.concat([best_bert, best_distil])

selected

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
selected_models = {}
for run in logs.values():
    score = max(epoch["eval_f1_macro"] for epoch in run["metrics"])
    name = run["model_name"]
    if name not in selected_models or score > selected_models[name][0]:
        selected_models[name] = (score, run)


predictions = {}
for model_name, (_, run) in selected_models.items():
    checkpoint_path = run["best_checkpoint"]
    
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path
    )
    ds_test = tokenize_dataset(X_test, y_test, checkpoint_path)

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=f"evaluation/{model_name}",
            per_device_eval_batch_size=16,
            report_to="none",
        ),
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )
    predictions[model_name] = trainer.predict(ds_test)

In [ ]:
results = []
for model_name, prediction in predictions.items():
    metrics = prediction.metrics
    metrics["model_name"] = model_name
    results.append(metrics)

test_metrics = pd.DataFrame(results)

test_metrics



In [ ]:
import time
import joblib
import numpy as np


baseline_model = joblib.load("models/logistic_regression.pkl")

start = time.perf_counter()

y_pred = baseline_model.predict(X_test)

runtime = time.perf_counter() - start

y_prob = baseline_model.predict_proba(X_test)

baseline_metrics = {
    "model_name": "baseline",
    **{f"test_{key}": value for key, value in classification_metrics(y_test, y_pred, probabilities=y_prob).items()},
    "test_runtime": runtime,
    "test_samples_per_second": len(X_test) / runtime,
    "test_steps_per_second": 0,
}
results.append(baseline_metrics)


test_metrics = pd.DataFrame(results)

In [ ]:
test_metrics

In [ ]:

results = []
for model_name, prediction in predictions.items():
    metrics = prediction.metrics
    metrics["model_name"] = model_name
    results.append(metrics)


In [ ]:
fig, axes = plt.subplots(2,4, figsize = (16,8), sharey = False)

metrics = ['test_log_loss', 'test_accuracy', 'test_precision_macro', 'test_recall_macro',
           'test_f1_macro', 'test_runtime', 'test_samples_per_second', 'test_steps_per_second']


for index, metric in enumerate(metrics):
    i = index // 4
    j = index % 4

    axes[i, j].bar(test_metrics["model_name"], test_metrics[metric])
    axes[i, j].set_title(metric)
    if index < 5:
        axes[i, j].set_ylim(0,1)
    #axes[i, j].tick_params(axis="x", labelrotation=90)

fig.suptitle("Test metrics")
plt.show()



The transformer models are significantly better at inference than the baseline. The distilbert model is marginally better at inference, but not in any significant way. Inference speed, however, is significantly better, therefore this will be the chosen model for gradio deployment.

test logloss is still quite high, the model shows signs of overconfidence. Therefore, we will use temperature scaling to minimize validation logloss, then well evaluate on the test set.

In [ ]:
ds_val = tokenize_dataset(X_val, y_val, "distilbert-base-uncased")

In [ ]:
import torch
import torch.nn.functional as F
from scipy.optimize import minimize_scalar

checkpoint_path = selected_models["distilbert-base-uncased"][1]["best_checkpoint"]
model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint_path
)
ds_val = tokenize_dataset(X_val, y_val, "distilbert-base-uncased")

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        per_device_eval_batch_size=16,
        report_to="none",
    ),
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
# Predictions from the selected checkpoint
output = trainer.predict(ds_val)

logits = torch.tensor(output.predictions, dtype=torch.float64)
labels = torch.tensor(output.label_ids, dtype=torch.long)

#search using a log-transform of the temperature for a more convenient search space

def calibration_loss(log_temperature):
    temperature = np.exp(log_temperature)

    loss = F.cross_entropy(
        logits / temperature,
        labels,
    )

    return loss.item()

result = minimize_scalar(
    calibration_loss,
    bounds=(-5, 5),
    method="bounded",
)

temperature = np.exp((result.x))

print(f"Optimal temperature: {temperature:.10f}")
print(f"Original validation loss: {calibration_loss(0):.4f}")
print(f"Calibrated validation loss: {calibration_loss(result.x):.4f}")

funny, the optimal temperature is close to euler's constant, optimal log-temperature is close to one. We'll round to 2.71 for convenience. Let's compute test loss to see how much it improved

In [ ]:
from scipy.special import softmax
from sklearn.metrics import brier_score_loss

TEMP = 2.71

test_output = trainer.predict(ds_test)

logits = torch.tensor(test_output.predictions, dtype=torch.float64)
y_true= torch.tensor(test_output.label_ids, dtype=torch.long)

original = softmax(logits, axis=1)
calibrated = softmax(logits / TEMP, axis=1)

for name, probs in [
    ("Original", original),
    ("Calibrated", calibrated),
]:
    print(name)
    metrics = classification_metrics(
        y_true, np.argmax(probs, axis=1), probabilities=probs
    )
    print("Log loss:", metrics["log_loss"])
    print("Brier score:", brier_score_loss(y_true, probs[:, 1]))
    print("Mean confidence:", probs.max(axis=1).mean())